# Figure SX: SO-EPT Coupling and SWCF - Historical vs. PiControl 

This analysis sanity checks that SO-EPT Coupling and SWCF calculate from piControl as similar to those derived from historical. This analysis is in response to a reviewer comment

In [1]:
import xarray as xr
import xcdat as xc
import numpy as np
import xskillscore as xscore
import matplotlib.pyplot as plt
import os

# Utils

In [2]:
def fix_coords(data):
    data = data.bounds.add_bounds("X")
    data = data.bounds.add_bounds("Y")
    data = data.bounds.add_bounds("T")
    data = xc.swap_lon_axis(data, to=(-180, 180))
    return data

def remove_land_full(ds, var="skt"):
    ds = xc.swap_lon_axis(ds, to=(-180, 180))
    from global_land_mask import globe
    # Set land to NaN
    lon_grid,lat_grid = np.meshgrid(ds.lon, ds.lat)
    globe_land_mask = globe.is_land(lat_grid,lon_grid)
    globe_land_mask_nd = np.tile(globe_land_mask,(ds[var].shape[0],ds[var].shape[1], 1,1))
    ds_no_land = xr.where(globe_land_mask_nd==True,np.nan,ds[var]) 
    return ds_no_land


def get_rolling_timeseries(data: xr.DataArray, window: int = 12*30, step: int = 12, save: bool = False, name: str = "", gradient: bool = False) -> xr.DataArray:
    """Get the rolling timeseries of a dataset (optionally calculate gradient)

    Args:
        data (xr.DataArray): Input data.
        window (int, optional): Size of the rolling window. Defaults to 12*30.
        step (int, optional): Step size for each window. Defaults to 12.
        gradient (bool, optional): Calculate gradient if True, mean if False. Defaults to False.

    Returns:
        xr.DataArray: Resulting timeseries with same shape as input.
    """
    
    # Prepare output array with the same shape as input data
    new_time_size = 1 + (len(data.time) - window) // step
    rolling_shape = list(data.shape)
    rolling_shape[data.get_axis_num('time')] = new_time_size

    # Prepare output array with the adjusted shape
    rolling = np.full(rolling_shape, np.nan)
    time_idx = xr.DataArray(np.arange(window), dims="time")

    data = data.chunk({"time": -1})
    # Loop to compute rolling statistics over time dimension
    for j, i in enumerate(range(0, len(data.time) - window, step)):
        if gradient:
            rolling[:,j] = window*xscore.linslope(time_idx, data.isel(time=slice(i, i+window)), dim='time', skipna=True).values
        else:
            rolling[:,j] = data.isel(time=slice(i, i+window)).mean(dim="time").values

    # Convert to xarray
    da = xr.DataArray(rolling, dims=data.dims, coords={**data.coords, 'time': np.arange(rolling.shape[1])})

    return da


def lagged_regression(ts_so_ssts, ts_ept_ssts, lags, models):
    reg_coeffs, rvalues, pvalues = [], [], []
    for lag in lags:
        rvalues.append(xscore.pearson_r(ts_so_ssts.shift(time=lag), ts_ept_ssts, dim="time", skipna=True))
        reg_coeffs.append(xscore.linslope(ts_so_ssts.shift(time=lag), ts_ept_ssts, dim="time", skipna=True))
        pvalues.append(xscore.pearson_r_eff_p_value(ts_so_ssts.shift(time=lag), ts_ept_ssts, dim="time", skipna=True))


    reg_coeffs = xr.Dataset({'reg': (['lags', 'model'], np.array(reg_coeffs))}, coords={'model': models, 'lags': lags})
    rvalues = xr.Dataset({'rvalues': (['lags', 'model'], np.array(rvalues))}, coords={'model': models, 'lags': lags})
    pvalues = xr.Dataset({'pvalues': (['lags', 'model'], np.array(pvalues))}, coords={'model': models, 'lags': lags})
    # Combine to xr.Datasets into one
    regs_dataset = xr.merge([rvalues, reg_coeffs, pvalues])
    return regs_dataset


def get_triangle(tos, latmin: float = -38.75, latmax: float = -1.25, lonmin: float = -178.75, lonmax: float = -71.25, RES: float = 2.5):
    DY = latmax - latmin
    DX = lonmax - lonmin 
    dx = RES*round(DX/DY)
    dy = RES

    print(f"For each latitude step of {dy} degrees, longitude step is {dx}")

    latcoords = np.arange(latmax, latmin-dy, -dy)
    loncoords = np.arange(lonmin, lonmax+dx, dx)
    lonraw = np.arange(lonmin, lonmax+dx, RES)

    ctos = tos.sel(lon=slice(lonmin, lonmax), lat=slice(latmin, latmax))
    nmodel, ntime, _, nlon = ctos.shape
    # print(ctos)

    for i, clon in enumerate(lonraw):
        j = np.where(clon == loncoords)[0]

        if i == nlon: break

        # print("j prior: ", j)
        if len(j) == 0: 
            j = jold
        else: 
            j = j[0]
             
        # print("j: ", j)
        nlats = int(len(latcoords) - j) # nlats below diag
        # print("nlats: ", nlats)
        ctos[:,:,:nlats,i] = np.full((nmodel, ntime, nlats), np.nan) 
        
        jold = j
    
    return ctos

# Preprocess

In [3]:
# Loading PiControl data
def load_cmip_data(fp):
    """Load CMIP data from a given file path."""
    # Open the dataset with xarray
    cmip_monthly_ssts = xr.open_dataset(fp, chunks="auto")
    # Fix coordinates and remove land
    cmip_monthly_ssts = fix_coords(remove_land_full(cmip_monthly_ssts, var="tos").to_dataset())
    # Get the EPT SSTs 
    cmip_ept_ssts = fix_coords(get_triangle(cmip_monthly_ssts.tos.copy(deep=True)).to_dataset()).spatial.average("tos").temporal.departures("tos", "month")["tos"]
    # Get the SO SSTs
    cmip_so_ssts = cmip_monthly_ssts.sel(lon=slice(-180, -75), lat=slice(-70, -50)).spatial.average("tos").temporal.departures("tos", "month")["tos"]

    return cmip_so_ssts, cmip_ept_ssts

so6_pi, ept6_pi = load_cmip_data("/home/espinosa10/tropical_pacific_clouds/data/piControl/tos_mon_1850-2100_CMIP5_piControl.nc")
so6_hi, ept6_hi = load_cmip_data("/home/espinosa10/tropical_pacific_clouds/data/historical/tos_mon_1850-2100_CMIP6_historical.nc")

For each latitude step of 2.5 degrees, longitude step is 7.5


/home/espinosa10/miniconda3/envs/swc/lib/python3.11/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing with an out-of-order index is generating 150 times more chunks
  return self.array[key]
/home/espinosa10/miniconda3/envs/swc/lib/python3.11/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing with an out-of-order index is generating 150 times more chunks
  return self.array[key]


For each latitude step of 2.5 degrees, longitude step is 7.5


/home/espinosa10/miniconda3/envs/swc/lib/python3.11/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing with an out-of-order index is generating 165 times more chunks
  return self.array[key]
/home/espinosa10/miniconda3/envs/swc/lib/python3.11/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing with an out-of-order index is generating 165 times more chunks
  return self.array[key]


In [ ]:
def calculate_coupling_depending_on_time(so_ssts, ept_ssts, models):
    min_year, max_year = 1, 30
    months = np.arange(1, 12)
    years = np.arange(min_year*12, (max_year+1)*12, 12)

    time = [*months, *years]
    reg_coeffs, rvalues, pvalues, variance_so, variance_ept = [], [], [], [], []
    for year in time:
        ts_so_ssts = get_rolling_timeseries(so_ssts, window=year, step=12, gradient=False)
        ts_ept_ssts = get_rolling_timeseries(ept_ssts, window=year, step=12, gradient=False)

        variance_ept.append(ts_ept_ssts.std(dim="time"))
        variance_so.append(ts_so_ssts.std(dim="time"))
        rvalues.append(xscore.pearson_r(ts_so_ssts, ts_ept_ssts, dim="time", skipna=True))
        pvalues.append(xscore.pearson_r_eff_p_value(ts_so_ssts, ts_ept_ssts, dim="time", skipna=True))
        reg_coeffs.append(xscore.linslope(ts_so_ssts, ts_ept_ssts, dim="time", skipna=True))

    reg_coeffs = xr.Dataset({'reg': (['years', 'model'], np.array(reg_coeffs))}, coords={'model': models, 'years': time})
    rvalues = xr.Dataset({'rvalues': (['years', 'model'], np.array(rvalues))}, coords={'model': models, 'years': time})
    pvalues = xr.Dataset({'pvalues': (['years', 'model'], np.array(pvalues))}, coords={'model': models, 'years': time})
    variance_so = xr.Dataset({'variance_so': (['years', 'model'], np.array(variance_so))}, coords={'model': models, 'years': time})
    variance_ept = xr.Dataset({'variance_ept': (['years', 'model'], np.array(variance_ept))}, coords={'model': models, 'years': time})

    # Combine to xr.Datasets into one
    regs_dataset = xr.merge([rvalues, reg_coeffs, pvalues, variance_so, variance_ept])
    return regs_dataset

# Historical Coupling
if not os.path.exists("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling_historical.nc"):
    so_ept_coupling_hi = calculate_coupling_depending_on_time(so6_hi, ept6_hi, models=ept6_hi.model)
    so_ept_coupling_hi.to_netcdf("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling_historical.nc")
else: 
    so_ept_coupling_hi = xr.open_dataset("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling_historical.nc")

# PiControl Coupling
if not os.path.exists("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling.nc"):
    so_ept_coupling_pi = calculate_coupling_depending_on_time(so6_pi, ept6_pi, models=ept6_pi.model)
    so_ept_coupling_pi.to_netcdf("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling.nc")
else: 
    so_ept_coupling_pi = xr.open_dataset("/home/espinosa10/SO-EP-teleconnection/Data/so_ept_coupling.nc")

# Figure SX: Historical vs. PiControl